# Gmail Bot - Send Emails with Attachments from Excel

This notebook reads company names and emails from an Excel file and sends personalized emails with attachments.

## Setup Requirements
1. Enable 2-Factor Authentication on your Gmail account
2. Generate an App Password: Google Account → Security → App Passwords
3. Upload your Excel file and attachment(s) to Colab

In [ ]:
# Install required packages
!pip install pandas openpyxl

In [ ]:
import smtplib
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import os
import time

In [ ]:
# Upload files in Colab
from google.colab import files

print("Upload your Excel file (with columns: 'email', 'company_name')")
uploaded = files.upload()
excel_file = list(uploaded.keys())[0]

print("\nUpload your attachment file(s)")
attachments = files.upload()
attachment_files = list(attachments.keys())

In [ ]:
# Configuration
SENDER_EMAIL = "your_email@gmail.com"  # Your Gmail address
APP_PASSWORD = "xxxx xxxx xxxx xxxx"   # Your 16-char App Password (no spaces needed)

# Email template
SUBJECT = "Partnership Opportunity with {company_name}"
BODY_TEMPLATE = """
Dear {company_name} Team,

I hope this email finds you well.

I am reaching out to discuss a potential partnership opportunity.

Please find the attached document for more details.

Best regards,
Your Name
"""

In [ ]:
def read_excel_data(file_path):
    """Read email and company data from Excel file."""
    df = pd.read_excel(file_path)
    # Normalize column names to lowercase
    df.columns = df.columns.str.lower().str.strip()
    
    # Check for required columns
    required = ['email', 'company_name']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    
    return df[['email', 'company_name']].dropna()

# Preview the data
df = read_excel_data(excel_file)
print(f"Found {len(df)} recipients:")
df.head()

In [ ]:
def create_email_with_attachment(sender, recipient, subject, body, attachment_paths):
    """Create email message with attachments."""
    msg = MIMEMultipart()
    msg['From'] = sender
    msg['To'] = recipient
    msg['Subject'] = subject
    
    # Attach body
    msg.attach(MIMEText(body, 'plain'))
    
    # Attach files
    for file_path in attachment_paths:
        with open(file_path, 'rb') as f:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(f.read())
            encoders.encode_base64(part)
            part.add_header(
                'Content-Disposition',
                f'attachment; filename="{os.path.basename(file_path)}"'
            )
            msg.attach(part)
    
    return msg

In [ ]:
def send_emails(df, sender_email, app_password, attachment_files, delay=2):
    """Send emails to all recipients in dataframe."""
    # Connect to Gmail SMTP
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(sender_email, app_password.replace(' ', ''))
    
    sent = 0
    failed = []
    
    for idx, row in df.iterrows():
        try:
            email = row['email']
            company = row['company_name']
            
            # Personalize email
            subject = SUBJECT.format(company_name=company)
            body = BODY_TEMPLATE.format(company_name=company)
            
            # Create and send
            msg = create_email_with_attachment(
                sender_email, email, subject, body, attachment_files
            )
            server.sendmail(sender_email, email, msg.as_string())
            
            sent += 1
            print(f"✓ Sent to {company} ({email})")
            
            # Delay to avoid rate limiting
            time.sleep(delay)
            
        except Exception as e:
            failed.append({'email': email, 'error': str(e)})
            print(f"✗ Failed: {email} - {e}")
    
    server.quit()
    
    print(f"\n--- Summary ---")
    print(f"Sent: {sent}/{len(df)}")
    if failed:
        print(f"Failed: {len(failed)}")
    
    return sent, failed

In [ ]:
# Send emails (uncomment to run)
# sent, failed = send_emails(df, SENDER_EMAIL, APP_PASSWORD, attachment_files)

# Test with first recipient only
test_df = df.head(1)
print("Testing with first recipient:")
print(test_df)
# sent, failed = send_emails(test_df, SENDER_EMAIL, APP_PASSWORD, attachment_files)

---
## Alternative: Selenium Approach (Not Recommended for Colab)

Selenium requires browser setup and is less reliable. Use only if you need to interact with Gmail's web interface directly.

In [ ]:
# Selenium setup for Colab (only if needed)
# !apt-get update
# !apt-get install -y chromium-chromedriver
# !pip install selenium

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

# def setup_driver():
#     options = webdriver.ChromeOptions()
#     options.add_argument('--headless')
#     options.add_argument('--no-sandbox')
#     options.add_argument('--disable-dev-shm-usage')
#     return webdriver.Chrome(options=options)

# Note: Gmail blocks most automated logins via Selenium due to security.
# The SMTP approach above is much more reliable.